## Setup

In [ ]:
from pathlib import Path
import importlib.metadata
import importlib
import os
import subprocess
import sys

PROJECT = Path('/content/NFPLight_code')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'codex/colab-experiments',
                    'https://github.com/n01r1r/NFPLight_code.git', str(PROJECT)], check=True)
os.chdir(PROJECT)
remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], text=True).strip()
if remote.removesuffix('.git') != 'https://github.com/n01r1r/NFPLight_code':
    raise RuntimeError('Existing clone belongs to another repository')
subprocess.run(['git', 'fetch', '-q', 'origin', 'codex/colab-experiments'], check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if head != subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], text=True).strip():
    raise RuntimeError('Existing clone is older. Keep any running trainer; open this notebook in a fresh GPU runtime.')
for name in ('nfplight', 'colab_tools'):
    module = sys.modules.get(name)
    if module is not None and not Path(module.__file__).resolve().is_relative_to(PROJECT):
        raise RuntimeError('Restart runtime: another clone is already imported')
sys.path.insert(0, str(PROJECT))
loaded = {name:getattr(sys.modules[name], '__version__', None)
          for name in ('einops', 'numpy', 'psutil', 'matplotlib', 'tqdm', 'packaging') if name in sys.modules}
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
if any(version is not None and version != importlib.metadata.version(name) for name, version in loaded.items()):
    raise RuntimeError('A loaded dependency changed. Restart the runtime and rerun.')
from packaging.version import Version
import torch
if not Version('2.1') <= Version(torch.__version__) < Version('3'):
    raise RuntimeError('Preinstalled CUDA PyTorch >=2.1,<3 required')
get_ipython().run_line_magic('matplotlib', 'inline')
import colab_tools
importlib.reload(colab_tools)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU available:', torch.cuda.is_available())


## Drive data

In [ ]:
from google.colab import drive
from colab_tools import attach_drive_dataset, relative_path, relocate_manifest

DRIVE_FOLDER = 'MyDrive/MatSynth_PRIMA_Fabric'
drive.mount('/content/drive')
relative_path(DRIVE_FOLDER)
data_root = attach_drive_dataset(Path('/content/drive') / DRIVE_FOLDER, 'data/MatSynth_PRIMA_Fabric')
MANIFEST = Path('work/train_val_manifest.jsonl')
inventory = relocate_manifest(data_root / 'manifests/fabric_leather_training_manifest.jsonl', data_root, MANIFEST,
    expected_counts={'train/Fabric':358, 'train/Leather':208, 'val/Fabric':98, 'val/Leather':66})
print(inventory['materials'], '| GB:', round(inventory['total_bytes'] / 1e9, 3))


## NFPLight 64M original · Train

In [ ]:
from colab_tools import launch_training, monitor

RUN_ROOT = Path('runs/nfplight_01')
training_process = launch_training('configs/nfplight_32_noquant.json', MANIFEST, RUN_ROOT)
monitor(RUN_ROOT, training_process)


## Live view

In [ ]:
from colab_tools import monitor

monitor(RUN_ROOT, globals().get('training_process'))
